# スタッキング検証(メモリ節約版) ― LGBM に TCN 予測を特徴入力

v10 の best 特徴セット(順方向+逆方向, `cmp_` 除外)に TCN の OOF 予測 `tcn_pred` を1列足し、
GroupKFold(5) by well で **overall RMSE** を測る。ベースライン(tcnなし)は v10 の既知値と比較する
(再実行しない=メモリ/時間節約)。

メモリ対策(RAM 20GB制約):
- parquet を pandas 全列 DataFrame 化しない。**pyarrow で列チャンク読み → float32 の1配列**に直書き。
- LightGBM は **native API + `Dataset.subset`** を使い、fold毎の巨大コピーを避ける(binは1回・共有)。
- base(tcnなし)は **再実行せず** v10 の (2) 値と比較。

リーク安全性: `tcn_pred = oof_tcn.npy` は坑井グループOOF → 特徴がリークせず val 指標は不偏。

In [1]:
import time
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error

PROJECT  = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
PARQUET  = PROJECT / "v10" / "artifacts_v10" / "data" / "train.parquet"
OOF_TCN  = PROJECT / "artifacts_v12" / "oof_tcn.npy"
N_SPLITS = 5

# v10 (2) 順方向+逆方向 の既知ベースライン(再実行しない)
BASELINE_OVERALL  = 7.5537
BASELINE_PERWELL  = 5.7349

NON_FEAT = {"well", "id", "target"}

# native 用パラメータ(v10 と同等)
PARAMS = dict(
    boosting_type="gbdt", num_leaves=255, min_child_samples=15,
    bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8,
    lambda_l2=3.0, lambda_l1=0.05, objective="regression", metric="rmse",
    verbosity=-1, num_threads=0, max_bin=255, learning_rate=0.030, seed=123,
    device_type="cpu",
)
NUM_ROUND = 5000
EARLY_STOP = 200

In [2]:
# frugal ロード: base(fwd_bwd)特徴 + tcn_pred を float32 の1配列に
all_cols = pq.read_schema(PARQUET).names
base_cols = [c for c in all_cols if c not in NON_FEAT and not c.startswith("cmp_")]  # = v10 (2)
F = len(base_cols)
meta = pq.read_table(PARQUET, columns=["well", "target"]).to_pandas()
n = len(meta)
y = meta["target"].to_numpy(np.float32)
groups = meta["well"].to_numpy()
print(f"rows={n}  wells={meta['well'].nunique()}  base_features={F} (+tcn_pred)")

X = np.empty((n, F + 1), np.float32)        # 最後の列に tcn_pred
CHUNK = 64
for i in range(0, F, CHUNK):
    cols = base_cols[i:i+CHUNK]
    sub = np.array(pq.read_table(PARQUET, columns=cols).to_pandas().to_numpy(np.float32))
    sub[~np.isfinite(sub)] = np.nan
    X[:, i:i+len(cols)] = sub
    del sub
tcn = np.load(OOF_TCN).astype(np.float32)
assert len(tcn) == n
X[:, F] = tcn
feat_names = base_cols + ["tcn_pred"]
print("X:", X.shape, X.dtype)

rows=3783989  wells=773  base_features=627 (+tcn_pred)
X: (3783989, 628) float32


In [ ]:
# native API + subset で メモリ節約 CV(base + tcn_pred)
full = lgb.Dataset(X, label=y, feature_name=feat_names, free_raw_data=False)
oof = np.full(n, np.nan, np.float32)
gkf = GroupKFold(n_splits=N_SPLITS)
t0 = time.perf_counter()
for k, (tr, va) in enumerate(gkf.split(X, y, groups)):
    tr_ds = full.subset(tr); va_ds = full.subset(va)
    booster = lgb.train(PARAMS, tr_ds, num_boost_round=NUM_ROUND, valid_sets=[va_ds],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
    oof[va] = booster.predict(X[va], num_iteration=booster.best_iteration).astype(np.float32)
    print(f"  fold {k}: rmse={root_mean_squared_error(y[va], oof[va]):.4f}  it={booster.best_iteration}")
    del tr_ds, va_ds, booster
print(f"done {time.perf_counter()-t0:.0f}s")

c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\lightgbm\basic.py:2535: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


  fold 0: rmse=6.9464  it=870


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\lightgbm\basic.py:2535: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


  fold 1: rmse=6.9568  it=949


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\lightgbm\basic.py:2535: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


  fold 2: rmse=7.2493  it=3382


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\lightgbm\basic.py:2535: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


In [ ]:
import pandas as pd
overall = float(root_mean_squared_error(y, oof))
tmp = pd.DataFrame({"well": groups, "e2": (oof - y) ** 2})
per_well = float(np.sqrt(tmp.groupby("well")["e2"].mean()).mean())
np.save(PROJECT / "artifacts_v12" / "oof_stack_lgbm_tcn.npy", oof)

print("==== stacking 結果 ====")
print(f"base (v10 (2), tcnなし)   : overall={BASELINE_OVERALL:.4f}  per_well={BASELINE_PERWELL:.4f}")
print(f"base + tcn_pred           : overall={overall:.4f}  per_well={per_well:.4f}")
print(f"  Δoverall = {overall-BASELINE_OVERALL:+.4f}   (マイナスならスタッキング有効)")
print(f"  参考: 加重ブレンド best = 7.3924")